<a href="https://colab.research.google.com/github/aadi9210004-byte/Python_Projects/blob/main/banking_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

from abc import ABC, abstractmethod
from decimal import Decimal, InvalidOperation
from datetime import datetime
import csv
import os


# ABSTRACT CLASS


class Record(ABC):

    @abstractmethod
    def display(self):
        pass


class BankAccount(Record):

    def __init__(self, account_number, holder_name,
                 account_type, balance=Decimal("0.00")):

        self.__account_number = account_number
        self.__holder_name = holder_name
        self.__account_type = account_type
        self.__balance = Decimal(balance).quantize(
            Decimal("0.01")
        )

    # Properties

    @property
    def account_number(self):
        return self.__account_number

    @property
    def holder_name(self):
        return self.__holder_name

    @property
    def account_type(self):
        return self.__account_type

    @property
    def balance(self):
        return self.__balance

    # Deposit money

    def deposit(self, amount):

        amount = Decimal(amount)

        if amount <= 0:
            return False

        self.__balance += amount
        return True

    # Withdraw money

    def withdraw(self, amount):

        amount = Decimal(amount)

        if amount <= 0:
            return False

        if amount > self.__balance:
            return False

        self.__balance -= amount
        return True

    # Display account

    def display(self):

        print("-" * 45)
        print("Account Number :", self.account_number)
        print("Account Holder :", self.holder_name)
        print("Account Type   :", self.account_type)
        print("Balance        : Rs.", self.balance)
        print("-" * 45)



class BankManagementSystem:

    def __init__(self):

        self.accounts = []
        self.transactions = []

        self.load_data()



    def find_account(self, account_number):

        for account in self.accounts:

            if account.account_number == account_number:
                return account

        return None


    def record_transaction(self, account_number,
                            transaction_type, amount,
                            balance_after):

        transaction = {
            "Date": datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
            "Account Number": account_number,
            "Type": transaction_type,
            "Amount": str(amount),
            "Balance After": str(balance_after)
        }

        self.transactions.append(transaction)



    def create_account(self, account_number,
                       holder_name, account_type,
                       initial_deposit):

        if self.find_account(account_number):
            print("Error: Account number already exists.")
            return

        if initial_deposit < 0:
            print("Initial deposit cannot be negative.")
            return

        account = BankAccount(
            account_number,
            holder_name,
            account_type,
            initial_deposit
        )

        self.accounts.append(account)

        self.record_transaction(
            account_number,
            "Account Opened",
            initial_deposit,
            account.balance
        )

        print("Account created successfully!")

        self.save_data()


    def deposit_money(self, account_number, amount):

        account = self.find_account(account_number)

        if account is None:
            print("Account not found.")
            return

        if amount <= 0:
            print("Deposit amount must be positive.")
            return

        account.deposit(amount)

        self.record_transaction(
            account_number,
            "Deposit",
            amount,
            account.balance
        )

        print("Deposit successful!")
        print("New Balance: Rs.", account.balance)

        self.save_data()



    def withdraw_money(self, account_number, amount):

        account = self.find_account(account_number)

        if account is None:
            print("Account not found.")
            return

        if amount <= 0:
            print("Withdrawal amount must be positive.")
            return

        if not account.withdraw(amount):
            print("Insufficient balance.")
            return

        self.record_transaction(
            account_number,
            "Withdrawal",
            amount,
            account.balance
        )

        print("Withdrawal successful!")
        print("Remaining Balance: Rs.", account.balance)

        self.save_data()

    # ------------------------------------------------------
    # CHECK BALANCE
    # ------------------------------------------------------

    def check_balance(self, account_number):

        account = self.find_account(account_number)

        if account is None:
            print("Account not found.")
            return

        account.display()

    def display_accounts(self):

        if not self.accounts:
            print("No accounts available.")
            return

        print("\nALL BANK ACCOUNTS")

        for account in self.accounts:
            account.display()


    def transaction_history(self, account_number):

        if self.find_account(account_number) is None:
            print("Account not found.")
            return

        history = [
            transaction
            for transaction in self.transactions
            if transaction["Account Number"] == account_number
        ]

        if not history:
            print("No transactions found.")
            return

        print("\nTRANSACTION HISTORY")


        for transaction in history:

            print("Date:", transaction["Date"])
            print("Type:", transaction["Type"])
            print("Amount: Rs.", transaction["Amount"])
            print(
                "Balance After: Rs.",
                transaction["Balance After"]
            )



    def save_accounts_csv(self):

        with open(
            "accounts.csv", "w",
            newline="", encoding="utf-8"
        ) as file:

            writer = csv.writer(file)

            writer.writerow([
                "Account Number",
                "Holder Name",
                "Account Type",
                "Balance"
            ])

            for account in self.accounts:

                writer.writerow([
                    account.account_number,
                    account.holder_name,
                    account.account_type,
                    str(account.balance)
                ])



    def save_transactions_csv(self):

        with open(
            "transactions.csv", "w",
            newline="", encoding="utf-8"
        ) as file:

            fieldnames = [
                "Date",
                "Account Number",
                "Type",
                "Amount",
                "Balance After"
            ]

            writer = csv.DictWriter(
                file, fieldnames=fieldnames
            )

            writer.writeheader()
            writer.writerows(self.transactions)

    def save_data(self):

        try:

            self.save_accounts_csv()
            self.save_transactions_csv()
            self.save_txt()

            print("All records saved successfully.")

        except OSError as error:

            print("Error saving data:", error)



    def save_txt(self):

        with open(
            "bank_records.txt", "w", encoding="utf-8"
        ) as file:

            file.write("BANK ACCOUNT MANAGEMENT SYSTEM\n")


            file.write("ACCOUNT RECORDS\n")

            for account in self.accounts:

                file.write(
                    f"Account Number: {account.account_number}\n"
                )

                file.write(
                    f"Holder Name: {account.holder_name}\n"
                )

                file.write(
                    f"Account Type: {account.account_type}\n"
                )

                file.write(
                    f"Balance: Rs. {account.balance}\n"
                )

                file.write("-" * 60 + "\n")

            file.write("\nTRANSACTION HISTORY\n")

            for transaction in self.transactions:

                file.write(str(transaction) + "\n")

    # ------------------------------------------------------
    # LOAD SAVED DATA
    # ------------------------------------------------------

    def load_data(self):

        if os.path.exists("accounts.csv"):

            try:

                with open(
                    "accounts.csv", "r",
                    newline="", encoding="utf-8"
                ) as file:

                    reader = csv.DictReader(file)
                    loaded_accounts = []

                    for row in reader:

                        balance = Decimal(row["Balance"])

                        if balance < 0:
                            raise ValueError(
                                "Negative account balance in file."
                            )

                        account = BankAccount(
                            row["Account Number"],
                            row["Holder Name"],
                            row["Account Type"],
                            balance
                        )

                        if any(
                            item.account_number
                            == account.account_number
                            for item in loaded_accounts
                        ):
                            raise ValueError(
                                "Duplicate account number in file."
                            )

                        loaded_accounts.append(account)

                    self.accounts = loaded_accounts

            except (OSError, ValueError, KeyError) as error:

                print("Could not load accounts:", error)

        if os.path.exists("transactions.csv"):

            try:

                with open(
                    "transactions.csv", "r",
                    newline="", encoding="utf-8"
                ) as file:

                    reader = csv.DictReader(file)

                    self.transactions = list(reader)

                    for transaction in self.transactions:

                        if self.find_account(
                            transaction["Account Number"]
                        ) is None:
                            raise ValueError(
                                "Transaction references an "
                                "unknown account."
                            )

                        amount = Decimal(transaction["Amount"])
                        balance = Decimal(
                            transaction["Balance After"]
                        )

                        if amount < 0 or balance < 0:
                            raise ValueError(
                                "Invalid transaction amount."
                            )

            except (OSError, ValueError, KeyError) as error:

                print("Could not load transactions:", error)
                self.transactions = []

        print("Bank system initialized.")


# ==========================================================
# INPUT HELPERS
# ==========================================================

def get_text(prompt):

    while True:

        value = input(prompt).strip()

        if value:
            return value

        print("This field cannot be empty.")


def get_money(prompt):

    while True:

        try:

            value = Decimal(input(prompt).strip())

            if not value.is_finite():
                print("Enter a valid finite amount.")
                continue

            if value < 0:
                print("Amount cannot be negative.")
                continue

            if value != value.quantize(Decimal("0.01")):
                print("Enter no more than two decimal places.")
                continue

            return value

        except InvalidOperation:

            print("Enter a valid monetary amount.")



# MAIN PROGRAM------------------------------------------------------------

bank = BankManagementSystem()


while True:


    print("     BANK ACCOUNT MANAGEMENT SYSTEM")
    print("=" * 45)

    print("1. Create Account")
    print("2. Deposit Money")
    print("3. Withdraw Money")
    print("4. Check Balance")
    print("5. Transaction History")
    print("6. Display All Accounts")
    print("7. Save Data")
    print("8. Exit")



    choice = input("Enter your choice: ").strip()

    try:

        if choice == "1":

            print("\nCREATE ACCOUNT")

            account_number = get_text("Account Number: ")
            holder_name = get_text("Account Holder Name: ")

            print("1. Savings")
            print("2. Current")

            account_choice = input(
                "Select Account Type: "
            ).strip()

            if account_choice == "1":
                account_type = "Savings"

            elif account_choice == "2":
                account_type = "Current"

            else:
                print("Invalid account type.")
                continue

            initial_deposit = get_money(
                "Initial Deposit (Rs.): "
            )

            bank.create_account(
                account_number,
                holder_name,
                account_type,
                initial_deposit
            )

        elif choice == "2":

            account_number = get_text("Account Number: ")
            amount = get_money("Deposit Amount (Rs.): ")

            bank.deposit_money(account_number, amount)

        elif choice == "3":

            account_number = get_text("Account Number: ")
            amount = get_money("Withdrawal Amount (Rs.): ")

            bank.withdraw_money(account_number, amount)

        elif choice == "4":

            account_number = get_text("Account Number: ")

            bank.check_balance(account_number)

        elif choice == "5":

            account_number = get_text("Account Number: ")

            bank.transaction_history(account_number)

        elif choice == "6":

            bank.display_accounts()

        elif choice == "7":

            bank.save_data()

        elif choice == "8":

            bank.save_data()

            print("Thank you for using the Bank System.")
            break

        else:

            print("Invalid choice. Select a number from 1 to 8.")

    except (ValueError, InvalidOperation) as error:

        print("Invalid input:", error)

    except OSError as error:

        print("File operation failed:", error)




Bank system initialized.
     BANK ACCOUNT MANAGEMENT SYSTEM
1. Create Account
2. Deposit Money
3. Withdraw Money
4. Check Balance
5. Transaction History
6. Display All Accounts
7. Save Data
8. Exit
Enter your choice: 1

CREATE ACCOUNT
Account Number: 1001
Account Holder Name: Rahul Sen
1. Savings
2. Current
Select Account Type: 1
Initial Deposit (Rs.): 5500
Account created successfully!
All records saved successfully.
     BANK ACCOUNT MANAGEMENT SYSTEM
1. Create Account
2. Deposit Money
3. Withdraw Money
4. Check Balance
5. Transaction History
6. Display All Accounts
7. Save Data
8. Exit
Enter your choice: 7
All records saved successfully.
     BANK ACCOUNT MANAGEMENT SYSTEM
1. Create Account
2. Deposit Money
3. Withdraw Money
4. Check Balance
5. Transaction History
6. Display All Accounts
7. Save Data
8. Exit
Enter your choice: 1

CREATE ACCOUNT
Account Number: 1002
Account Holder Name: Priya Das
1. Savings
2. Current
Select Account Type: 2
Initial Deposit (Rs.): 18270
Account creat